# Chapter 6 — pandas I: Tables of the Indian Economy

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch06_pandas_state_economies.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 6.1 The state accounts

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from indiadata import mospi_nas

gsdp = mospi_nas("GSDP")
gsdp.head()

In [ ]:
print("rows, columns:", gsdp.shape)
gsdp.dtypes

In [ ]:
gsdp.nunique()

In [ ]:
gsdp = gsdp.drop(columns=["indicator", "frequency", "unit"])
gsdp.head(3)

## 6.2 Series, DataFrames and the index

In [ ]:
real = gsdp["constant_price"]
print(type(real))
real.head()

In [ ]:
panel = gsdp.set_index(["state", "fy"])
panel.head()

In [ ]:
panel.loc[("Maharashtra", 2023), "constant_price"]

## 6.3 Selecting rows and columns

In [ ]:
gsdp[["state", "year", "constant_price"]].head(3)

In [ ]:
gsdp.iloc[:3, :2]

In [ ]:
panel.loc["Kerala"].loc[2018:2022, ["year", "constant_price"]]

In [ ]:
big = gsdp[(gsdp["fy"] == 2023) & (gsdp["current_price"] > 2_000_000)]
big[["state", "current_price"]]

In [ ]:
south = ["Andhra Pradesh", "Karnataka", "Kerala", "Tamil Nadu", "Telangana"]
gsdp[gsdp["state"].isin(south) & (gsdp["fy"] == 2023)][["state", "constant_price"]]

## 6.4 Sorting and new columns

In [ ]:
y23 = gsdp[gsdp["fy"] == 2023].copy()
y23["lakh_crore"] = y23["current_price"] / 1e5     # 1 lakh crore = 100,000 crore
top = y23.sort_values("lakh_crore", ascending=False)
top[["state", "lakh_crore"]].head(8).round(1)

In [ ]:
y23["share_%"] = 100 * y23["current_price"] / y23["current_price"].sum()
y23.nlargest(5, "share_%")[["state", "share_%"]].round(1)

In [ ]:
listed = set(gsdp["state"])
india = ["Assam", "Haryana", "Nagaland", "Ladakh", "Lakshadweep",
         "Dadra & Nagar Haveli and Daman & Diu"]
print("Not in MoSPI's GSDP table:", [s for s in india if s not in listed])

In [ ]:
gdp = mospi_nas("GDP")
india_gdp = gdp.set_index("fy")["current_price"]
states_sum = gsdp.groupby("fy")["current_price"].sum()
coverage = pd.DataFrame({"sum of 30 states": states_sum,
                         "all-India GDP": india_gdp}).loc[2011:2024]
coverage["states / India, %"] = 100 * coverage.iloc[:, 0] / coverage.iloc[:, 1]
coverage.loc[[2011, 2015, 2019, 2023, 2024]].round(1)

## 6.5 Summary statistics with `describe`

In [ ]:
y23[["current_price", "constant_price"]].describe().round(0)

## 6.6 Find the gaps before they find you

In [ ]:
gsdp.groupby("fy")["constant_price"].count().tail(4)

In [ ]:
latest = gsdp[gsdp["fy"] == 2025]
print("Missing 2025-26:", latest.loc[latest["constant_price"].isna(), "state"].tolist())

## 6.7 Group, summarise, reshape

In [ ]:
summary = (gsdp.groupby("state")["constant_price"]
               .agg(["count", "min", "max", "mean"]))
summary.sort_values("max", ascending=False).head().round(0)

In [ ]:
span = gsdp.dropna(subset=["constant_price"]).groupby("state").agg(
    first_year=("year", "first"),
    last_year=("year", "last"),
    years=("fy", "count"),
)
span["last_year"].value_counts()

In [ ]:
REGION = {
    "Jammu & Kashmir": "North", "Himachal Pradesh": "North", "Punjab": "North",
    "Chandigarh": "North", "Uttarakhand": "North", "Delhi": "North",
    "Rajasthan": "North",
    "Uttar Pradesh": "Central", "Madhya Pradesh": "Central",
    "Chhattisgarh": "Central",
    "Bihar": "East", "Jharkhand": "East", "Odisha": "East",
    "West Bengal": "East", "Andaman & Nicobar Islands": "East",
    "Arunachal Pradesh": "North-East", "Manipur": "North-East",
    "Meghalaya": "North-East", "Mizoram": "North-East", "Sikkim": "North-East",
    "Tripura": "North-East",
    "Gujarat": "West", "Maharashtra": "West", "Goa": "West",
    "Andhra Pradesh": "South", "Karnataka": "South", "Kerala": "South",
    "Tamil Nadu": "South", "Telangana": "South", "Puducherry": "South",
}
gsdp["region"] = gsdp["state"].map(REGION)
assert gsdp["region"].notna().all(), "a state has no region"

by_region = (gsdp[gsdp["fy"].isin([2011, 2024])]
             .groupby(["region", "fy"])["constant_price"].sum()
             .unstack("fy"))
by_region["share 2011-12, %"] = 100 * by_region[2011] / by_region[2011].sum()
by_region["share 2024-25, %"] = 100 * by_region[2024] / by_region[2024].sum()
by_region.round(1)

In [ ]:
wide = gsdp.pivot(index="state", columns="fy", values="constant_price")
wide.iloc[:5, -5:]

In [ ]:
pd.pivot_table(gsdp[gsdp["fy"] >= 2022], index="region", columns="year",
               values="current_price", aggfunc="sum", margins=True).round(-3)

## 6.8 Growth within each state: `shift` and `pct_change`

In [ ]:
gsdp = gsdp.sort_values(["state", "fy"])
gsdp["previous"] = gsdp.groupby("state")["constant_price"].shift(1)
gsdp["growth_%"] = 100 * (gsdp["constant_price"] / gsdp["previous"] - 1)
gsdp[gsdp["state"] == "Bihar"][["year", "constant_price", "previous",
                                "growth_%"]].head(4).round(1)

In [ ]:
check = 100 * gsdp.groupby("state")["constant_price"].pct_change()
print("Largest difference:", (check - gsdp["growth_%"]).abs().max())

In [ ]:
covid = gsdp[gsdp["fy"] == 2020].sort_values("growth_%")
print(f"States with negative real growth in 2020-21: "
      f"{(covid['growth_%'] < 0).sum()} of {len(covid)}")
covid[["state", "growth_%"]].iloc[[0, 1, 2, -3, -2, -1]].round(1)

## 6.9 The compound annual growth rate

In [ ]:
START, END = 2011, 2024
n = END - START
real_wide = gsdp.pivot(index="state", columns="fy", values="constant_price")
nom_wide = gsdp.pivot(index="state", columns="fy", values="current_price")

growth = pd.DataFrame({
    "real CAGR %": 100 * ((real_wide[END] / real_wide[START]) ** (1 / n) - 1),
    "nominal CAGR %": 100 * ((nom_wide[END] / nom_wide[START]) ** (1 / n) - 1),
})
growth["rank (real)"] = growth["real CAGR %"].rank(ascending=False).astype(int)
growth = growth.sort_values("rank (real)")
growth.head(6).round(2)

In [ ]:
growth.tail(6).round(2)

In [ ]:
mean_growth = gsdp[gsdp["fy"].between(START + 1, END)].groupby("state")["growth_%"].mean()
compare = pd.DataFrame({"CAGR": growth["real CAGR %"],
                        "mean of yearly growth": mean_growth})
compare["difference"] = compare["mean of yearly growth"] - compare["CAGR"]
print(f"Mean of yearly growth exceeds CAGR in "
      f"{(compare['difference'] > 0).sum()} of {len(compare)} states; "
      f"largest gap {compare['difference'].max():.2f} points "
      f"({compare['difference'].idxmax()})")

In [ ]:
g_india = gdp.set_index("fy")["constant_price"]
india_cagr = 100 * ((g_india[END] / g_india[START]) ** (1 / n) - 1)
print(f"All-India real GDP CAGR, 2011-12 to 2024-25: {india_cagr:.2f}%")
print(f"States growing faster than India: "
      f"{(growth['real CAGR %'] > india_cagr).sum()} of {len(growth)}")

## 6.10 A ranking chart

In [ ]:
order = growth.sort_values("real CAGR %")
colors = ["#c55a11" if r in ("South", "West") else "#1f4e79"
          for r in order.index.map(REGION)]

fig, ax = plt.subplots(figsize=(7.5, 8.5))
ax.barh(order.index, order["real CAGR %"], color=colors)
ax.axvline(india_cagr, color="black", lw=1, ls="--")
ax.text(india_cagr + 0.08, 0.3, f"All-India GDP {india_cagr:.1f}%", fontsize=9)
ax.set_xlabel("Compound annual real growth, 2011–12 to 2024–25, %")
ax.set_title("Real GSDP growth by state, 2011–12 to 2024–25",
             loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="y", labelsize=8.5)
ax.margins(y=0.01)
fig.text(0.01, 0.005, "Orange: South and West regions; blue: others. Constant "
         "2011–12 prices. Haryana, Assam and Nagaland are not in the table.\n"
         "Source: MoSPI, National Accounts Statistics (GSDP, base 2011–12), "
         "via eSankhyiki API.", fontsize=8, color="grey")
plt.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

## 6.11 Growth paths: a line chart

In [ ]:
gsdp["index_2011"] = (100 * gsdp["constant_price"]
                      / gsdp.groupby("state")["constant_price"].transform("first"))
paths = gsdp.pivot(index="fy", columns="state", values="index_2011").loc[:END]

fast, slow = growth.index[0], growth.index[-1]
fig, ax = plt.subplots(figsize=(9, 4.8))
for s in paths.columns:
    ax.plot(paths.index, paths[s], color="#d9d9d9", lw=0.8)
for s, c in [(fast, "#c55a11"), ("Maharashtra", "#1f4e79"), (slow, "#7f7f7f")]:
    ax.plot(paths.index, paths[s], color=c, lw=2.2)
    ax.text(END + 0.15, paths.loc[END, s], s, color=c, fontsize=9, va="center")
ax.axvspan(2019.6, 2020.4, color="#9db7d5", alpha=0.3)
ax.text(2020, ax.get_ylim()[1] * 0.97, "2020–21", ha="center", fontsize=8,
        color="#1f4e79")
ax.set_xticks(range(START, END + 1, 2))
ax.set_xticklabels([f"{y}–{str(y + 1)[2:]}" for y in range(START, END + 1, 2)])
ax.set_xlim(START, END + 2.6)
ax.set_ylabel("Real GSDP, 2011–12 = 100")
ax.set_title("Real GSDP paths of 30 states and UTs", loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.03, "Grey lines: other states. Source: MoSPI, National "
         "Accounts Statistics (GSDP, constant 2011–12 prices).",
         fontsize=8, color="grey")
plt.show()

## 6.12 What is a fair comparison?

In [ ]:
growth["deflator CAGR %"] = 100 * ((1 + growth["nominal CAGR %"] / 100)
                                   / (1 + growth["real CAGR %"] / 100) - 1)
growth["rank (nominal)"] = growth["nominal CAGR %"].rank(ascending=False).astype(int)
moved = (growth["rank (nominal)"] - growth["rank (real)"]).abs()
print(f"Deflator growth ranges from {growth['deflator CAGR %'].min():.1f}% "
      f"to {growth['deflator CAGR %'].max():.1f}% a year")
print(f"States whose rank changes by 5 or more places: {(moved >= 5).sum()}")
growth.loc[moved.sort_values(ascending=False).index[:5],
           ["real CAGR %", "nominal CAGR %", "rank (real)", "rank (nominal)"]].round(2)

In [ ]:
nsdp = mospi_nas("NSDP")
both = gsdp.merge(nsdp[["state", "fy", "current_price"]],
                  on=["state", "fy"], suffixes=("", "_nsdp"))
both["NSDP / GSDP %"] = 100 * both["current_price_nsdp"] / both["current_price"]
both[both["fy"] == 2023]["NSDP / GSDP %"].describe()[["min", "50%", "max"]].round(1)

## 6.13 Answering the question

In [ ]:
size = real_wide[END] / 1e5
answer = growth[["real CAGR %", "rank (real)"]].assign(
    **{"real GSDP 2024-25, ₹ lakh crore": size})
pd.concat([answer.head(5), answer.tail(5)]).round(2)

In [ ]:
print(f"Fastest: {growth.index[0]} ({growth['real CAGR %'].iloc[0]:.2f}% a year)")
print(f"Slowest: {growth.index[-1]} ({growth['real CAGR %'].iloc[-1]:.2f}% a year)")
print(f"Median state: {growth['real CAGR %'].median():.2f}% a year")
ratio_fast = (1 + growth["real CAGR %"].iloc[0] / 100) ** n
ratio_slow = (1 + growth["real CAGR %"].iloc[-1] / 100) ** n
print(f"Over {n} years the fastest economy grew {ratio_fast:.2f}-fold, "
      f"the slowest {ratio_slow:.2f}-fold")

## Key ideas

## Exercises

## Data citation

## Further reading